In [1]:
import torch

from test_attn_order_eval import AttnOrderEval   # signal-agnostic ranking eval over a [T, L] per-step score + order


class MarginOrderEval(AttnOrderEval):

    def _margin(self, logits):   # logits [T, L, V] -> p1 - p2 (top-1 prob minus top-2 prob) [T, L]
        logits = logits.to(torch.float64)                            # match the float64 softmax convention; chunk over T if memory-bound
        lse = torch.logsumexp(logits, dim=-1)                        # [T, L]  log-partition (full vocab scan)
        top2 = logits.topk(2, dim=-1).values                        # [T, L, 2]  rank 0 = largest logit
        p1 = (top2[..., 0] - lse).exp()                             # [T, L]  top-1 prob
        p2 = (top2[..., 1] - lse).exp()                             # [T, L]  top-2 prob
        return p1 - p2                                              # [T, L]
    # end

    def __init__(self, logits, order):                              # logits [T, L, V], order [T] long
        assert logits.dim() == 3, "logits.dim(): {} == 3 false".format(logits.dim())

        score = self._margin(logits)                                # [T, L]  per-step, per-position margin
        super().__init__(score, order)                              # reuse geometry + recall_at_h / pr_auc / ndcg_at_h
    # end

    # def __init__(self, margin, order):                              # logits [T, L, V], order [T] long
    #     score = self._margin(margin)                                # [T, L]  per-step, per-position margin
    #     super().__init__(score, order)                              # reuse geometry + recall_at_h / pr_auc / ndcg_at_h
    # # end


    def get_margin(self):
        return self.attn   # the base stores the ranking score here (generic [T, L] score slot)
    # end


if __name__ == "__main__":
    torch.manual_seed(0)
    T, L, V = 64, 64, 50
    order = torch.randperm(L)

    step_of = torch.full((L,), -1, dtype=torch.long)
    step_of[order] = torch.arange(T)
    gap = step_of.view(1, L) - torch.arange(T).view(T, 1)          # [T, L]

    # perfect: sharper peak (=> larger p1 - p2) for sooner-unmasked candidates
    peak = torch.where(gap > 0, 5.0 / gap.clamp_min(1).float(), torch.zeros(1))   # [T, L]
    logits_perfect = torch.zeros(T, L, V)
    logits_perfect[..., 0] = peak
    logits_random = torch.randn(T, L, V)

    def summ(x):
        return "{:.3f} (n={})".format(x.nanmean().item(), int((~x.isnan()).sum()))
    # end

    for name, lg in [("perfect", logits_perfect), ("random ", logits_random)]:
        ev = MarginOrderEval(lg, order)
        print("{}  recall@5={}  pr_auc@5={}  ndcg@10={}".format(
            name, summ(ev.recall_at_h(5)), summ(ev.pr_auc(5)), summ(ev.ndcg_at_h(10))))
    # end

perfect  recall@5=1.000 (n=59)  pr_auc@5=1.000 (n=58)  ndcg@10=1.000 (n=62)
random   recall@5=0.207 (n=59)  pr_auc@5=0.279 (n=58)  ndcg@10=0.348 (n=62)
